In [19]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
# sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set path to datasets
DATA_DIR = Path("/home/tella/code/StellaRodriguesLallement/OSE_Project/Dataset_visualization/src/ose_core/data_ingestion/extracted_datasets")

# Check it exists
print(DATA_DIR.exists())

print("Libraries imported successfully!")

True
Libraries imported successfully!


In [20]:
# LOADING DATASETS

df_basic     = pd.read_csv(DATA_DIR / '01_company_basic_info.csv', dtype={'siren': str})
df_financial = pd.read_csv(DATA_DIR / '02_financial_data.csv', dtype={'siren': str})
df_workforce = pd.read_csv(DATA_DIR / '03_workforce_data.csv', dtype={'siren': str})
df_structure = pd.read_csv(DATA_DIR / '04_company_structure.csv', dtype={'siren': str})
df_flags     = pd.read_csv(DATA_DIR / '05_classification_flags.csv', dtype={'siren': str})
df_contacts  = pd.read_csv(DATA_DIR / '06_contact_metrics.csv', dtype={'siren': str})
df_kpi       = pd.read_csv(DATA_DIR / '07_kpi_data.csv', dtype={'siren': str})
df_signals   = pd.read_csv(DATA_DIR / '08_signals.csv', dtype={'siren': str})
df_articles  = pd.read_csv(DATA_DIR / '09_articles.csv', dtype={'siren': str})

print("All datasets loaded successfully!")


All datasets loaded successfully!


In [27]:
df_basic.columns

Index(['company_name', 'siren', 'siret', 'departement', 'resume_activite',
       'raison_sociale_keyword', 'raison_sociale', 'last_modified',
       'processedAt', 'updatedAt'],
      dtype='object')

In [ ]:
# BASIC FEATURES
df_basic_feat = df_basic[['company_name','siren', 'siret', 'departement']].copy()


In [38]:
df_financial.columns.tolist()

['company_name',
 'siren',
 'siret',
 'caConsolide',
 'caGroupe',
 'resultatExploitation',
 'dateConsolide',
 'kpi_2025_capital_social',
 'kpi_2025_evolution_ca',
 'kpi_2023_ca_france',
 'kpi_2023_ca_bilan',
 'kpi_2023_resultat_exploitation',
 'kpi_2023_capital_social',
 'kpi_2023_resultat_bilan',
 'kpi_2022_ca_france',
 'kpi_2022_ca_bilan',
 'kpi_2022_resultat_exploitation',
 'kpi_2022_capital_social',
 'kpi_2022_resultat_bilan',
 'kpi_2021_ca_france',
 'kpi_2021_ca_bilan',
 'kpi_2021_resultat_exploitation',
 'kpi_2021_capital_social',
 'kpi_2021_resultat_bilan',
 'kpi_2020_ca_france',
 'kpi_2020_ca_bilan',
 'kpi_2020_resultat_exploitation',
 'kpi_2020_capital_social',
 'kpi_2020_resultat_bilan',
 'kpi_2019_ca_france',
 'kpi_2019_ca_bilan',
 'kpi_2019_resultat_exploitation',
 'kpi_2019_capital_social',
 'kpi_2019_resultat_bilan',
 'kpi_2018_ca_bilan',
 'kpi_2018_resultat_exploitation',
 'kpi_2018_capital_social',
 'kpi_2018_resultat_bilan',
 'kpi_2017_ca_france',
 'kpi_2017_ca_bilan',

In [44]:
import re

# Identify raw variables (those not starting with "kpi_")
raw_cols = [col for col in df_financial.columns if not col.startswith("kpi_")]

# Identify kpi variables
kpi_cols = [col for col in df_financial.columns if col.startswith("kpi_")]

raw_cols

['company_name',
 'siren',
 'siret',
 'caConsolide',
 'caGroupe',
 'resultatExploitation',
 'dateConsolide']

In [45]:
kpi_cols

['kpi_2025_capital_social',
 'kpi_2025_evolution_ca',
 'kpi_2023_ca_france',
 'kpi_2023_ca_bilan',
 'kpi_2023_resultat_exploitation',
 'kpi_2023_capital_social',
 'kpi_2023_resultat_bilan',
 'kpi_2022_ca_france',
 'kpi_2022_ca_bilan',
 'kpi_2022_resultat_exploitation',
 'kpi_2022_capital_social',
 'kpi_2022_resultat_bilan',
 'kpi_2021_ca_france',
 'kpi_2021_ca_bilan',
 'kpi_2021_resultat_exploitation',
 'kpi_2021_capital_social',
 'kpi_2021_resultat_bilan',
 'kpi_2020_ca_france',
 'kpi_2020_ca_bilan',
 'kpi_2020_resultat_exploitation',
 'kpi_2020_capital_social',
 'kpi_2020_resultat_bilan',
 'kpi_2019_ca_france',
 'kpi_2019_ca_bilan',
 'kpi_2019_resultat_exploitation',
 'kpi_2019_capital_social',
 'kpi_2019_resultat_bilan',
 'kpi_2018_ca_bilan',
 'kpi_2018_resultat_exploitation',
 'kpi_2018_capital_social',
 'kpi_2018_resultat_bilan',
 'kpi_2017_ca_france',
 'kpi_2017_ca_bilan',
 'kpi_2017_resultat_exploitation',
 'kpi_2017_capital_social',
 'kpi_2017_resultat_bilan',
 'kpi_2016_ca_bil

In [ ]:
# Extract the KPI base name (ex: 'ca_bilan', 'resultat_exploitation', etc.)
def strip_year(col):
    return re.sub(r'kpi_\d{4}_', '', col)  # removes "kpi_2024_"

# Compare each raw column against matching KPI columns
results = {}

for raw in raw_cols:
    # find KPI columns that correspond to the raw variable
    raw_normalised = raw.lower().replace("consolide", "consolide").replace("exploitation", "exploitation")

    related_kpis = [k for k in kpi_cols if strip_year(k).lower() == raw_normalised]

    if not related_kpis:
        results[raw] = "⚠️ No corresponding KPI-year column found"
        continue

    # compare values
    match_info = {}
    for kcol in related_kpis:
        same = (df_financial[raw] == df_financial[kcol]).sum()
        total = df_financial.shape[0]
        match_ratio = same / total
        match_info[kcol] = round(match_ratio, 4)

    results[raw] = match_info

results


{'company_name': '⚠️ No corresponding KPI-year column found',
 'siren': '⚠️ No corresponding KPI-year column found',
 'siret': '⚠️ No corresponding KPI-year column found',
 'caConsolide': '⚠️ No corresponding KPI-year column found',
 'caGroupe': '⚠️ No corresponding KPI-year column found',
 'resultatExploitation': '⚠️ No corresponding KPI-year column found',
 'dateConsolide': '⚠️ No corresponding KPI-year column found'}

In [36]:
# Select only financial columns with at least 150 non-null values
min_non_null = 150

financial_selected_cols = (
    df_financial
        .drop(columns=['company_name', 'siren', 'siret'])      # keep only metrics
        .count()
        .loc[lambda x: x >= min_non_null]
        .index.tolist()
)

print("Selected FINANCIAL columns:", financial_selected_cols)

df_financial_feat = df_financial[['siren'] + financial_selected_cols].copy()


Selected FINANCIAL columns: ['caConsolide', 'caGroupe', 'resultatExploitation', 'dateConsolide', 'kpi_2022_ca_bilan', 'kpi_2022_capital_social', 'kpi_2022_resultat_bilan', 'kpi_2021_ca_bilan', 'kpi_2021_resultat_exploitation', 'kpi_2021_capital_social', 'kpi_2021_resultat_bilan', 'kpi_2020_ca_france', 'kpi_2020_ca_bilan', 'kpi_2020_resultat_exploitation', 'kpi_2020_capital_social', 'kpi_2020_resultat_bilan', 'kpi_2019_ca_france', 'kpi_2019_ca_bilan', 'kpi_2019_resultat_exploitation', 'kpi_2019_capital_social', 'kpi_2019_resultat_bilan', 'kpi_2018_ca_bilan', 'kpi_2018_resultat_exploitation', 'kpi_2018_capital_social', 'kpi_2018_resultat_bilan', 'kpi_2017_ca_france', 'kpi_2017_ca_bilan', 'kpi_2017_resultat_exploitation', 'kpi_2017_capital_social', 'kpi_2017_resultat_bilan', 'kpi_2016_ca_bilan', 'kpi_2016_resultat_bilan', 'kpi_2015_ca_bilan', 'kpi_2015_resultat_bilan', 'kpi_2014_ca_bilan', 'kpi_2014_resultat_bilan', 'kpi_2018_ca_france', 'kpi_2016_ca_france', 'kpi_2016_resultat_exploitati

In [37]:
df_financial_feat.head()

,siren,caConsolide,caGroupe,resultatExploitation,dateConsolide,kpi_2022_ca_bilan,kpi_2022_capital_social,kpi_2022_resultat_bilan,kpi_2021_ca_bilan,kpi_2021_resultat_exploitation,...,kpi_2015_resultat_bilan,kpi_2014_ca_bilan,kpi_2014_resultat_bilan,kpi_2018_ca_france,kpi_2016_ca_france,kpi_2016_resultat_exploitation,kpi_2016_capital_social,kpi_2013_ca_bilan,kpi_2013_resultat_bilan,kpi_2024_capital_social
0,000132066,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,015751530,0.0,0.0,76546.0,0.0,6247357.0,120000.0,182001.0,5296275.0,-62109.0,...,181450.0,6653070.0,388230.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,016450298,0.0,0.0,670860.0,0.0,8678668.0,257600.0,453579.0,7588920.0,429064.0,...,NaN,NaN,NaN,6954431.0,NaN,NaN,NaN,NaN,NaN,NaN
3,046580031,0.0,0.0,140333.0,0.0,NaN,350000.0,354077.0,NaN,NaN,...,140570.0,4068800.0,168950.0,NaN,3215827.0,-317.0,350000.0,3760470.0,199020.0,NaN
4,057504649,0.0,0.0,473736.0,0.0,NaN,1000000.0,180831.0,NaN,NaN,...,298258.0,6125829.0,115160.0,NaN,6489800.0,585559.0,1000000.0,5639506.0,90561.0,NaN


In [25]:
df_workforce.columns

Index(['company_name', 'siren', 'siret', 'effectif', 'effectifConsolide',
       'effectifEstime', 'effectifGroupe'],
      dtype='object')

In [62]:
# WORKFORCE FEATURES
df_workforce_feat = df_workforce[['siren', 'effectif']].copy()

# Convert to numeric (if needed)
# workforce_feat['effectif'] = pd.to_numeric(workforce_feat['effectif'], errors='coerce')


In [47]:
df_structure.columns

Index(['company_name', 'siren', 'siret', 'nbFilialesDirectes',
       'nbEtabSecondaire', 'nbMarques', 'hasGroupOwner', 'appartient_groupe',
       'nombre_etablissements_secondaires_inactifs'],
      dtype='object')

In [61]:
# STRUCTURE FEATURES

structure_selected = [
    'siren',
    'nbFilialesDirectes',
    'nbEtabSecondaire',
    'nbMarques',
    'hasGroupOwner',   # or 'appartient_groupe'
    'nombre_etablissements_secondaires_inactifs'
]

df_structure_feat = df_structure[structure_selected].copy()
df_structure_feat.head()


,siren,nbFilialesDirectes,nbEtabSecondaire,nbMarques,hasGroupOwner,nombre_etablissements_secondaires_inactifs
0,000132066,0.0,0.0,0.0,False,0
1,015751530,6.0,4.0,14.0,False,3
2,016450298,1.0,0.0,4.0,False,2
3,046580031,0.0,0.0,3.0,False,0
4,057504649,0.0,0.0,0.0,False,0


In [57]:
df_flags.columns

Index(['company_name', 'siren', 'siret', 'startup', 'radiee', 'entreprise_b2b',
       'entreprise_b2c', 'fintech', 'cac40', 'entreprise_familiale',
       'entreprise_familiale_ter', 'filtre_levee_fond', 'flag_type_entreprise',
       'hasMarques', 'hasESV1Contacts'],
      dtype='object')

In [58]:
df_flags.isnull().mean().sort_values(ascending=False)

siret                       0.064000
entreprise_familiale_ter    0.061333
company_name                0.000000
siren                       0.000000
startup                     0.000000
radiee                      0.000000
entreprise_b2b              0.000000
entreprise_b2c              0.000000
fintech                     0.000000
cac40                       0.000000
entreprise_familiale        0.000000
filtre_levee_fond           0.000000
flag_type_entreprise        0.000000
hasMarques                  0.000000
hasESV1Contacts             0.000000
dtype: float64

In [63]:
# FLAG FEATURES
df_flags_feat = df_flags[['siren','filtre_levee_fond', 'hasMarques']].copy()

df_flags_feat.head()

,siren,filtre_levee_fond,hasMarques
0,000132066,False,False
1,015751530,False,True
2,016450298,False,True
3,046580031,False,True
4,057504649,False,False


In [31]:
df_signals.columns

Index(['company_name', 'siren', 'siret', 'continent', 'country', 'departement',
       'publishedAt', 'isMain', 'type', 'createdAt', 'companies_count',
       'sirets_count'],
      dtype='object')

In [71]:
# Prepare signals
df_signals_feat = df_signals[['siren', 'type', 'createdAt']].copy()
df_signals_feat = df_signals_feat.rename(columns={'createdAt': 'signal_createdAt'})
df_signals_feat.head()

,siren,type,signal_createdAt
0,015751530,"{'code': 'K1', 'id': 32, 'label': 'Investissem...",2020-09-07T15:14:38+02:00
1,015751530,"{'code': 'L', 'id': 12, 'label': 'Levée de fon...",2020-09-07T15:14:12+02:00
2,015751530,"{'code': 'F', 'id': 6, 'label': ""Développement...",2016-09-20T10:45:13+02:00
3,015751530,"{'code': 'F', 'id': 6, 'label': ""Développement...",2018-04-05T11:16:18+02:00
4,015751530,"{'code': 'E', 'id': 5, 'label': 'Créations & o...",2018-04-05T11:15:32+02:00


In [43]:
signals_feat[type].type()

KeyError: <class 'pandas.core.frame.DataFrame'>

In [32]:
df_articles.columns

Index(['company_name', 'siren', 'siret', 'title', 'publishedAt', 'author',
       'signalsStatus', 'signalsType', 'country', 'sectors', 'cities',
       'sources', 'all_companies_count'],
      dtype='object')

In [52]:
df_articles.isnull().sum()


company_name            0
siren                   0
siret                   0
title                   0
publishedAt             0
author                 21
signalsStatus           0
signalsType             0
country                 1
sectors                12
cities                 88
sources                32
all_companies_count     0
dtype: int64

In [56]:
df_articles.isnull().mean().sort_values(ascending=False)

cities                 0.074576
sources                0.027119
author                 0.017797
sectors                0.010169
country                0.000847
company_name           0.000000
siren                  0.000000
siret                  0.000000
title                  0.000000
publishedAt            0.000000
signalsStatus          0.000000
signalsType            0.000000
all_companies_count    0.000000
dtype: float64

In [ ]:
# ARTICLES FEATURES
df_articles_feat = df_articles[['siren','filtre_levee_fond', 'hasMarques']].copy()

df_flags_feat.head()

In [49]:
# Load KPI data
df_kpi = pd.read_csv(DATA_DIR / '07_kpi_data.csv', dtype={'siren': str, 'siret': str})

print(f"Dataset shape: {df_kpi.shape}")
print(f"\nColumns: {list(df_kpi.columns)}")
display(df_kpi.head(5))

Dataset shape: (3779, 28)

Columns: ['company_name', 'siren', 'siret', 'year', 'fonds_propres', 'ca_france', 'date_cloture_exercice', 'duree_exercice', 'salaires_traitements', 'charges_financieres', 'impots_taxes', 'ca_bilan', 'resultat_exploitation', 'dotations_amortissements', 'capital_social', 'code_confidentialite', 'resultat_bilan', 'annee', 'effectif', 'effectif_sous_traitance', 'filiales_participations', 'evolution_ca', 'subventions_investissements', 'ca_export', 'evolution_effectif', 'participation_bilan', 'ca_consolide', 'resultat_net_consolide']


,company_name,siren,siret,year,fonds_propres,ca_france,date_cloture_exercice,duree_exercice,salaires_traitements,charges_financieres,...,effectif,effectif_sous_traitance,filiales_participations,evolution_ca,subventions_investissements,ca_export,evolution_effectif,participation_bilan,ca_consolide,resultat_net_consolide
0,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2023,2192166.0,6729652.0,2023-01-31,12.0,1394492.0,80993.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2022,1614077.0,6247357.0,2022-01-31,12.0,1327711.0,81469.0,...,35.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2021,1497114.0,5296275.0,2021-01-31,12.0,1318083.0,66111.0,...,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2020,1577275.0,5710890.0,2020-01-31,12.0,1380952.0,70953.0,...,45.0,18930.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,01575153000013,2019,1348804.0,5221375.0,2019-01-31,12.0,1230571.0,88389.0,...,43.0,15835.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
df_kpi.columns.tolist()

['company_name',
 'siren',
 'siret',
 'year',
 'fonds_propres',
 'ca_france',
 'date_cloture_exercice',
 'duree_exercice',
 'salaires_traitements',
 'charges_financieres',
 'impots_taxes',
 'ca_bilan',
 'resultat_exploitation',
 'dotations_amortissements',
 'capital_social',
 'code_confidentialite',
 'resultat_bilan',
 'annee',
 'effectif',
 'effectif_sous_traitance',
 'filiales_participations',
 'evolution_ca',
 'subventions_investissements',
 'ca_export',
 'evolution_effectif',
 'participation_bilan',
 'ca_consolide',
 'resultat_net_consolide']

In [ ]:
feature_cols = [
    'fonds_propres',
    'resultat_exploitation',
    #'effectif'
    #'recency'   # giving weight to fresh companies
]

# Keeping only modelling columns + identifiers
cols_to_keep = ['company_name', 'siren'] + feature_cols
df_kpi_feat = df_kpi[cols_to_keep].copy()

df_kpi_feat.head()

,company_name,siren,fonds_propres,resultat_exploitation,effectif
0,PAIN D'EPICES MULOT ET PETITJEAN,015751530,2192166.0,76546.0,NaN
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1614077.0,65908.0,35.0
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1497114.0,-62109.0,34.0
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1577275.0,93684.0,45.0
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1348804.0,-247638.0,43.0


In [70]:
df_kpi_feat.shape

(3779, 5)

In [72]:
# Creating dataset by merge
# Starting with basic features
df_merged = df_basic_feat.copy()

# Merge workforce
df_merged = df_merged.merge(df_workforce_feat, on='siren', how='left')

# Merge structure
df_merged = df_merged.merge(df_structure_feat, on='siren', how='left')

# Merge flags
df_merged = df_merged.merge(df_flags_feat, on='siren', how='left')

# Merge signals
df_merged = df_merged.merge(df_signals_feat, on='siren', how='left')


In [ ]:
df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2242 entries, 0 to 2241
Data columns (total 14 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   company_name                                2242 non-null   object 
 1   siren                                       2242 non-null   object 
 2   siret                                       2218 non-null   float64
 3   departement                                 2242 non-null   object 
 4   effectif                                    2219 non-null   float64
 5   nbFilialesDirectes                          2152 non-null   float64
 6   nbEtabSecondaire                            2152 non-null   float64
 7   nbMarques                                   2152 non-null   float64
 8   hasGroupOwner                               2242 non-null   bool   
 9   nombre_etablissements_secondaires_inactifs  2242 non-null   int64  
 10  filtre_levee

In [74]:
df_merged.head()

,company_name,siren,siret,departement,effectif,nbFilialesDirectes,nbEtabSecondaire,nbMarques,hasGroupOwner,nombre_etablissements_secondaires_inactifs,filtre_levee_fond,hasMarques,type,signal_createdAt
0,PROCONI,000132066,NaN,00,0.0,0.0,0.0,0.0,False,0,False,False,NaN,NaN
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'K1', 'id': 32, 'label': 'Investissem...",2020-09-07T15:14:38+02:00
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'L', 'id': 12, 'label': 'Levée de fon...",2020-09-07T15:14:12+02:00
3,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'F', 'id': 6, 'label': ""Développement...",2016-09-20T10:45:13+02:00
4,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1.575153e+12,21,35.0,6.0,4.0,14.0,False,3,False,True,"{'code': 'F', 'id': 6, 'label': ""Développement...",2018-04-05T11:16:18+02:00


In [76]:
df_merged.isnull().mean().sort_values(ascending=False)

type                                          0.048617
signal_createdAt                              0.048617
nbFilialesDirectes                            0.040143
nbEtabSecondaire                              0.040143
nbMarques                                     0.040143
siret                                         0.010705
effectif                                      0.010259
company_name                                  0.000000
siren                                         0.000000
departement                                   0.000000
hasGroupOwner                                 0.000000
nombre_etablissements_secondaires_inactifs    0.000000
filtre_levee_fond                             0.000000
hasMarques                                    0.000000
dtype: float64

In [79]:
df_merged.describe().T

,count,mean,std,min,25%,50%,75%,max
siret,2218.0,4.752240e+13,1.595535e+13,1.575153e+12,3.507062e+13,4.443239e+13,5.388667e+13,9.504515e+13
effectif,2219.0,6.282965e+01,6.880793e+01,0.000000e+00,3.500000e+01,3.500000e+01,7.500000e+01,3.750000e+02
nbFilialesDirectes,2152.0,5.631970e-01,1.733447e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00
nbEtabSecondaire,2152.0,1.940056e+00,5.740526e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,9.300000e+01
nbMarques,2152.0,1.056924e+01,1.924768e+01,0.000000e+00,0.000000e+00,2.000000e+00,1.100000e+01,8.100000e+01
nombre_etablissements_secondaires_inactifs,2242.0,2.174398e+00,4.374218e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,6.700000e+01


In [80]:
df_merged.shape

(2242, 14)